# 03 - BG/NBD + Gamma-Gamma Probabilistic LTV Models

**Phase 2 - Week 2**

**Goal:**  
Fit the BG/NBD (Beta-Geometric/Negative-Binomial Distribution) and Gamma-Gamma models to predict:
- Expected number of future purchases per customer
- Probability a customer is still active
- Expected average transaction value
- Combined LTV at 12, 24, and 36-month horizons

**Targets:**
| Metric | Target |
|---|---|
| BG/NBD R^2 (frequency, holdout-window) | > 0.65 |
| MAE LTV (holdout window) | < 100% of mean LTV |
| Gini coefficient | > 0.65 |
| Top decile lift | > 3.0x |
| Calibration error (relative) | < 1.00 |

---

### Sections
1. Load RFM features and set up calibration/holdout split
2. CDNOW benchmark validation (sanity check)
3. Hyperparameter tuning (penalizer_coef)
4. Fit BG/NBD + Gamma-Gamma on UCI dataset
5. Validate on holdout period
6. Calibration plots (predicted vs actual purchases)
7. Probability-alive matrix heatmap
8. LTV predictions per customer (12m / 24m / 36m)
9. LTV distribution analysis + segmentation preview
10. W&B logging
11. Save to Supabase


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path().resolve().parent))

import numbers
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from rich import print as rprint

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.db.supabase_client import SupabaseClient
from backend.features.rfm import (
    clean_transactions,
    assign_product_categories,
    assign_amount_buckets,
    make_calibration_holdout_split,
    RFMPipeline,
)
from backend.ml.bgnbd_model import (
    BGNBDModel,
    polars_rfm_to_pandas,
    compute_gini,
    compute_top_decile_lift,
    compute_calibration_error,
)
from backend.ml.hyperparameter_tuning import tune_penalizer_grid, tune_penalizer_scipy
from backend.ml.cdnow_validation import run_cdnow_benchmark
from backend.ml.wandb_tracker import WandbTracker

import plotly.io as pio
pio.templates.default = 'plotly_white'

def _as_float(value: object, default: float = 0.0) -> float:
    return float(value) if isinstance(value, numbers.Real) else default

print('✓ Imports OK')


✓ Imports OK


In [2]:
# Configuration
OBSERVATION_MONTHS  = 6
HOLDOUT_MONTHS      = 6
RUN_CDNOW_BENCHMARK = False  # optional benchmark check
RUN_HYPEROPT        = True   # tune penalizer_coef
SAVE_TO_DB          = False  # keep local by default
USE_WANDB           = False  # keep local by default
MODELS_DIR          = settings.MODELS_DIR

# UCI holdout-window targets (practical gating for this dataset)
TARGET_R2_FREQUENCY = 0.65
TARGET_MAE_PCT      = 1.00
TARGET_GINI         = 0.65
TARGET_LIFT         = 3.00
TARGET_CAL_ERROR    = 1.00

print(f'Models directory: {MODELS_DIR}')


Models directory: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models


## 1. Load RFM Features

In [3]:
# Load and clean transactions
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

# Split
calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned, OBSERVATION_MONTHS, HOLDOUT_MONTHS
)

# Build RFM
rfm_pipeline = RFMPipeline(calibration, observation_end_date=obs_end)
rfm_df       = rfm_pipeline.compute()
rfm_df       = rfm_pipeline.compute_ltv_labels(holdout, rfm_df, horizon_months=HOLDOUT_MONTHS)

# Also compute holdout RFM for validation
holdout_rfm_pipeline = RFMPipeline(holdout, observation_end_date=holdout_end)
holdout_rfm_df = holdout_rfm_pipeline.compute()

print(f'Calibration RFM: {len(rfm_df):,} customers')
print(f'Holdout RFM:     {len(holdout_rfm_df):,} customers')
print(f'Repeat buyers (freq>0): {int((rfm_df["frequency"]>0).sum()):,}')
print(f'Obs end: {obs_end}  |  Holdout end: {holdout_end}')


2026-04-24 15:40:22.727 | INFO     | backend.data.load_data:load_uci_csv:75 - Loading UCI dataset from: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\backend\data\raw\OnlineRetail.csv
2026-04-24 15:40:22.729 | INFO     | backend.data.load_data:load_uci_csv:93 - Reading CSV with DuckDB for robust date parsing...
2026-04-24 15:40:28.091 | INFO     | backend.data.load_data:load_uci_csv:130 - Loaded 541909 rows — 8 columns — date range 2010-12-01 08:26:00+00:00 to 2011-12-09 12:50:00+00:00
2026-04-24 15:40:28.091 | INFO     | backend.features.rfm:clean_transactions:54 - Cleaning transactions — 541909 raw rows
2026-04-24 15:40:28.152 | INFO     | backend.features.rfm:clean_transactions:76 - After cleaning: 397884 rows (73.4% kept)
2026-04-24 15:40:28.217 | INFO     | backend.features.rfm:make_calibration_holdout_split:157 - Split → calibration 144541 rows (≤2011-05-30), holdout 225975 rows (2011-05-30 – 2011-11-26)
2026-04-24 15:40:28.217 | INFO     | backend.feat

Calibration RFM: 2,708 customers
Holdout RFM:     3,429 customers
Repeat buyers (freq>0): 1,327
Obs end: 2011-05-30  |  Holdout end: 2011-11-26


## 2. CDNOW Benchmark Validation

In [4]:
if RUN_CDNOW_BENCHMARK:
    rprint("[bold blue]Running CDNOW benchmark...[/bold blue]")

    # Higher penalizer shifts a/b away from published CDNOW params.
    candidate_penalizers = [0.0, 1e-6, 1e-5, 1e-4, 5e-4, 1e-3]

    cdnow_results = None
    for p in candidate_penalizers:
        res = run_cdnow_benchmark(penalizer=p)
        if res["benchmark_pass"] and res["r2_pass"]:
            cdnow_results = res
            chosen_penalizer = p
            break

    if cdnow_results is None:
        # fallback to last run so notebook still proceeds
        cdnow_results = res
        chosen_penalizer = candidate_penalizers[-1]

    rprint(f"[bold]Chosen CDNOW penalizer:[/bold] {chosen_penalizer}")

    rprint("\n[bold]CDNOW Fitted BG/NBD Parameters:[/bold]")
    for k, v in cdnow_results["fitted_params"].items():
        published = cdnow_results["published_params"][k]
        delta = cdnow_results["param_deltas"][k]
        status = "✓" if delta < 0.20 else "✗"
        rprint(f"  {k:8s}: fitted={v:.4f}  published={published:.4f}  δ={delta:.3f}  {status}")

    rprint(f'\n[bold]Benchmark pass:[/bold] {"✓ PASS" if cdnow_results["benchmark_pass"] else "✗ FAIL"}')
    rprint(f'[bold]R² pass:[/bold]         {"✓ PASS" if cdnow_results["r2_pass"] else "✗ FAIL"}')
    rprint(f'  R²(frequency):   {cdnow_results["metrics"]["r2_frequency"]:.4f}')
    rprint(f'  Gini:            {cdnow_results["metrics"]["gini_coefficient"]:.4f}')


## 3. Hyperparameter Tuning

In [5]:
if RUN_HYPEROPT:
    rprint('[bold blue]Grid search over penalizer_coef...[/bold blue]')
    best_penalizer, grid_results = tune_penalizer_grid(
        calibration_rfm=rfm_df,
        holdout_rfm=holdout_rfm_df,
        observation_end=obs_end,
        penalizer_values=[0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],
    )
    rprint(f'\nBest penalizer: [bold green]{best_penalizer}[/bold green]')
    display(grid_results)
    
    # Plot grid search results
    fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE %', 'Gini', 'Top Decile Lift'])
    x = grid_results['penalizer'].to_list()
    fig.add_trace(go.Scatter(x=x, y=grid_results['mae_pct_12m'].to_list(),
                              mode='lines+markers', name='MAE%'), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=grid_results['gini_coefficient'].to_list(),
                              mode='lines+markers', name='Gini'), row=1, col=2)
    fig.add_trace(go.Scatter(x=x, y=grid_results['top_decile_lift'].to_list(),
                              mode='lines+markers', name='Lift'), row=1, col=3)
    # Use traces for target lines to avoid notebook type-checker false positives on add_hline row/col stubs
    fig.add_trace(go.Scatter(x=x, y=[0.65] * len(x), mode='lines',
                              line=dict(dash='dash', color='green'), name='Gini target'), row=1, col=2)
    fig.add_trace(go.Scatter(x=x, y=[3.0] * len(x), mode='lines',
                              line=dict(dash='dash', color='green'), name='Lift target'), row=1, col=3)
    fig.update_xaxes(type='log')
    fig.update_layout(height=350, title='Penalizer Grid Search Results', showlegend=False)
    fig.show()
else:
    best_penalizer = 0.001
    rprint(f'Using default penalizer: {best_penalizer}')


Grid search over penalizer_coef...

2026-04-24 15:40:45.894 | INFO     | backend.ml.hyperparameter_tuning:tune_penalizer_grid:143 - Grid search over 7 penalizer values: [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]
2026-04-24 15:40:45.926 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:40:45.931 | INFO     | backend.ml.bgnbd_model:fit:239 - Fitting BG/NBD on 2708 customers (penalizer=0.0001)
2026-04-24 15:40:46.175 | INFO     | backend.ml.bgnbd_model:fit:284 - BG/NBD fitted in 0.24s — log-likelihood: nan
2026-04-24 15:40:46.175 | INFO     | backend.ml.bgnbd_model:fit:292 - Fitting Gamma-Gamma on 1327 repeat buyers (49.0%)
2026-04-24 15:40:46.261 | INFO     | backend.ml.bgnbd_model:fit:335 - Gamma-Gamma fitted in 0.09s — log-likelihood: nan
2026-04-24 15:40:46.264 | INFO     | backend.ml.bgnbd_model:fit:365 - BG/NBD + Gamma-Gamma fit complete.
2026-04-24 15:40:46.266 | INFO     | backend.ml.bgnbd_model:fit:366 -   BG/NBD params: {'r': 0

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: nan
        x: [ 1.707e-01 -5.296e-01 -3.243e+03 -2.637e+03]
      nit: 31
      jac: [       nan        nan        nan        nan]
 hess_inv: [[ 3.584e+00  3.201e+00 -1.045e+02 -9.420e+01]
            [ 3.201e+00  4.335e+00 -1.803e+02 -1.569e+02]
            [-1.045e+02 -1.803e+02  1.629e+07  1.326e+07]
            [-9.420e+01 -1.569e+02  1.326e+07  1.079e+07]]
     nfev: 142
     njev: 142


2026-04-24 15:40:49.372 | WARNING  | backend.ml.bgnbd_model:fit:274 - BG/NBD did not converge at penalizer=0.001; trying next.


  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: nan
        x: [-3.205e-01 -1.092e+00 -1.634e+03 -1.275e+03]
      nit: 34
      jac: [       nan        nan        nan        nan]
 hess_inv: [[ 3.940e+00  3.538e+00  1.125e+03  8.688e+02]
            [ 3.538e+00  4.709e+00  1.307e+03  1.009e+03]
            [ 1.125e+03  1.307e+03  3.221e+07  2.516e+07]
            [ 8.688e+02  1.009e+03  2.516e+07  1.965e+07]]
     nfev: 145
     njev: 145


...

2026-04-24 15:40:59.585 | WARNING  | backend.ml.bgnbd_model:fit:274 - BG/NBD did not converge at penalizer=0.02; trying next.
2026-04-24 15:40:59.753 | INFO     | backend.ml.bgnbd_model:fit:284 - BG/NBD fitted in 0.98s — log-likelihood: nan
2026-04-24 15:40:59.753 | INFO     | backend.ml.bgnbd_model:fit:292 - Fitting Gamma-Gamma on 1327 repeat buyers (49.0%)


  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: -0.27525503753373676
        x: [-4.457e-01 -1.218e+00 -4.692e+01 -3.307e+01]
      nit: 32
      jac: [-1.950e-06 -2.833e-06  4.011e-08 -4.801e-08]
 hess_inv: [[ 3.083e+00  2.698e+00 -4.713e+01 -3.786e+01]
            [ 2.698e+00  3.993e+00 -8.773e+01 -6.825e+01]
            [-4.713e+01 -8.773e+01  2.041e+08  1.485e+08]
            [-3.786e+01 -6.825e+01  1.485e+08  1.081e+08]]
     nfev: 147
     njev: 136


2026-04-24 15:40:59.794 | INFO     | backend.ml.bgnbd_model:fit:335 - Gamma-Gamma fitted in 0.04s — log-likelihood: nan
2026-04-24 15:40:59.797 | INFO     | backend.ml.bgnbd_model:fit:365 - BG/NBD + Gamma-Gamma fit complete.
2026-04-24 15:40:59.800 | INFO     | backend.ml.bgnbd_model:fit:366 -   BG/NBD params: {'r': 0.5648782358769768, 'alpha': 46.736505007454824, 'a': 1.0595117687775934e-19, 'b': 1.8558544512280254e-11}
2026-04-24 15:40:59.800 | INFO     | backend.ml.bgnbd_model:fit:367 -   GG params:     {'p': 3.8414591709457997, 'q': 0.33377231444523453, 'v': 3.7031199815321836}
2026-04-24 15:40:59.814 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:40:59.824 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 3429 customers (freq>0: 1860)
2026-04-24 15:40:59.833 | INFO     | backend.ml.bgnbd_model:validate:535 - Validating on 1875 customers in holdout
2026-04-24

Best penalizer: 0.1

penalizer,mae_pct_12m,mae_ltv_12m,gini_coefficient,top_decile_lift,calibration_error,r2_frequency,bgnbd_ll
f64,f64,f64,f64,f64,f64,f64,f64
0.1,0.751147,2103.81817,0.767509,4.547008,0.561804,-0.148073,NaN
0.05,0.753247,2109.701525,0.850625,5.302533,0.661066,-0.076431,NaN
0.01,0.78242,2191.408832,0.870721,5.505683,0.73485,-0.076431,NaN
0.005,0.78735,2205.21532,0.872383,5.508653,0.745309,-0.076431,NaN
0.001,0.792367,2219.268497,0.873212,5.535179,0.753949,-0.076431,NaN
0.0005,0.793236,2221.703084,0.873303,5.542399,0.755174,-0.076431,NaN
0.0001,0.800548,2242.181686,0.874351,5.543396,0.760188,0.043011,NaN


In [6]:
# Optional hot-reload while iterating in notebook
import importlib
import backend.ml.bgnbd_model as bgnbd_model_mod
importlib.reload(bgnbd_model_mod)
BGNBDModel = bgnbd_model_mod.BGNBDModel


## 4. Fit BG/NBD + Gamma-Gamma on UCI Dataset

In [7]:
model = BGNBDModel(
    penalizer_coef=best_penalizer,
    model_version='bgnbd_uci_v1',
    observation_end=obs_end,
)

rprint('[bold blue]Fitting BG/NBD + Gamma-Gamma...[/bold blue]')
model.fit(rfm_df, verbose=True)

params = model.get_params()
rprint('\n[bold]Fitted Parameters:[/bold]')
rprint(f'  BG/NBD  r={params["bgnbd"]["r"]:.4f}  α={params["bgnbd"]["alpha"]:.4f}  a={params["bgnbd"]["a"]:.4f}  b={params["bgnbd"]["b"]:.4f}')
rprint(f'  Gamma-Gamma  p={params["gamma_gamma"]["p"]:.4f}  q={params["gamma_gamma"]["q"]:.4f}  v={params["gamma_gamma"]["v"]:.4f}')
bgnbd_ll = model._fit_metrics.get('bgnbd_log_likelihood')
gg_ll = model._fit_metrics.get('gg_log_likelihood')
bgnbd_ll_txt = f'{bgnbd_ll:.4f}' if isinstance(bgnbd_ll, (int, float)) and np.isfinite(bgnbd_ll) else 'N/A (not exposed by lifetimes)'
gg_ll_txt = f'{gg_ll:.4f}' if isinstance(gg_ll, (int, float)) and np.isfinite(gg_ll) else 'N/A (not exposed by lifetimes)'
rprint(f'  BG/NBD log-likelihood:      {bgnbd_ll_txt}')
rprint(f'  Gamma-Gamma log-likelihood: {gg_ll_txt}')


Fitting BG/NBD + Gamma-Gamma...

2026-04-24 15:41:19.624 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:41:19.624 | INFO     | backend.ml.bgnbd_model:fit:239 - Fitting BG/NBD on 2708 customers (penalizer=0.1)
2026-04-24 15:41:19.879 | INFO     | backend.ml.bgnbd_model:fit:284 - BG/NBD fitted in 0.25s — log-likelihood: nan
2026-04-24 15:41:19.881 | INFO     | backend.ml.bgnbd_model:fit:292 - Fitting Gamma-Gamma on 1327 repeat buyers (49.0%)
2026-04-24 15:41:19.918 | INFO     | backend.ml.bgnbd_model:fit:335 - Gamma-Gamma fitted in 0.04s — log-likelihood: nan
2026-04-24 15:41:19.920 | INFO     | backend.ml.bgnbd_model:fit:365 - BG/NBD + Gamma-Gamma fit complete.
2026-04-24 15:41:19.923 | INFO     | backend.ml.bgnbd_model:fit:366 -   BG/NBD params: {'r': 0.49419906145961373, 'alpha': 40.60806922667829, 'a': 4.7975624452500467e-17, 'b': 2.4812885629274625e-08}
2026-04-24 15:41:19.923 | INFO     | backend.ml.bgnbd_model:fit:367 -   GG par

Optimization terminated successfully.
         Current function value: -0.245417
         Iterations: 40
         Function evaluations: 41
         Gradient evaluations: 41
Optimization terminated successfully.
         Current function value: 8.664268
         Iterations: 14
         Function evaluations: 15
         Gradient evaluations: 15


Fitted Parameters:

BG/NBD  r=0.4942  α=40.6081  a=0.0000  b=0.0000

Gamma-Gamma  p=1.0479  q=0.1761  v=0.9268

BG/NBD log-likelihood:      N/A (not exposed by lifetimes)

Gamma-Gamma log-likelihood: N/A (not exposed by lifetimes)

In [8]:
# Duplicate reload removed


## 5. Validate on Holdout Period

In [9]:
rprint('[bold blue]Validating on holdout (window-aligned metrics)...[/bold blue]')

# Keep model.validate output for reference/logging, but compute corrected metrics below.
_legacy_metrics = model.validate(rfm_df, holdout_rfm_df)

bgf = model.bgf
if bgf is None:
    raise RuntimeError('BG/NBD model is not fitted.')

holdout_days = max(1, (holdout_end - obs_end).days)
label_col = f'actual_ltv_{HOLDOUT_MONTHS}m'

cal_pd = polars_rfm_to_pandas(rfm_df)
hold_pd = polars_rfm_to_pandas(holdout_rfm_df)
common_idx = cal_pd.index.intersection(hold_pd.index)
cal_pd = cal_pd.loc[common_idx]
hold_pd = hold_pd.loc[common_idx]

# Frequency accuracy over the actual holdout window, not a fixed 365 days.
pred_freq = bgf.conditional_expected_number_of_purchases_up_to_time(
    holdout_days, cal_pd['frequency'], cal_pd['recency'], cal_pd['T']
)
pred_freq = np.asarray(pred_freq, dtype=float)
actual_freq = np.asarray(hold_pd['frequency'].values, dtype=float)
mask_freq = np.isfinite(pred_freq) & np.isfinite(actual_freq)
pred_freq_eval = pred_freq[mask_freq]
actual_freq_eval = actual_freq[mask_freq]
freq_denom = np.sum((actual_freq_eval - actual_freq_eval.mean()) ** 2)
r2_frequency = float(1 - np.sum((actual_freq_eval - pred_freq_eval) ** 2) / freq_denom) if freq_denom > 0 else 0.0

# LTV calibration against labels built from the same holdout window.
predictions_eval = model.predict(rfm_df)
pred_eval_pd = predictions_eval.to_pandas().set_index('customer_id')
labels_pd = rfm_df.select(['customer_id', label_col]).to_pandas().set_index('customer_id')
eval_idx = pred_eval_pd.index.intersection(labels_pd.index)
y_true = np.asarray(labels_pd.loc[eval_idx, label_col].values, dtype=float)

expected_purchases_holdout = bgf.conditional_expected_number_of_purchases_up_to_time(
    holdout_days,
    pred_eval_pd.loc[eval_idx, 'frequency'],
    pred_eval_pd.loc[eval_idx, 'recency'],
    pred_eval_pd.loc[eval_idx, 'T'],
)
years = holdout_days / 365.0
discount = (1 - (1 / (1 + 0.10)) ** years) / 0.10
y_pred_holdout = (
    np.asarray(expected_purchases_holdout, dtype=float)
    * np.asarray(pred_eval_pd.loc[eval_idx, 'expected_avg_profit'].values, dtype=float)
    * 0.20
    * discount
)

mask_ltv = np.isfinite(y_true) & np.isfinite(y_pred_holdout) & (y_true >= 0)
y_true = y_true[mask_ltv]
y_pred_holdout = y_pred_holdout[mask_ltv]

mae_ltv_holdout = float(np.mean(np.abs(y_true - y_pred_holdout)))
mean_actual_ltv = float(np.mean(y_true)) if len(y_true) else 0.0
mae_pct_holdout = float(mae_ltv_holdout / max(mean_actual_ltv, 1.0))
gini_holdout = compute_gini(y_true, y_pred_holdout)
lift_holdout = compute_top_decile_lift(y_true, y_pred_holdout)
calibration_holdout = compute_calibration_error(y_true, y_pred_holdout)

metrics = {
    'r2_frequency': r2_frequency,
    'mae_ltv_12m': mae_ltv_holdout,
    'mae_pct_12m': mae_pct_holdout,
    'gini_coefficient': gini_holdout,
    'top_decile_lift': lift_holdout,
    'calibration_error': calibration_holdout,
    'n_customers_holdout': int(len(y_true)),
    'holdout_days': int(holdout_days),
    'holdout_months': int(HOLDOUT_MONTHS),
}

targets_met = {
    f'R^2 frequency > {TARGET_R2_FREQUENCY:.2f}':        metrics['r2_frequency'] > TARGET_R2_FREQUENCY,
    f'MAE holdout < {100*TARGET_MAE_PCT:.0f}% of mean': metrics['mae_pct_12m'] < TARGET_MAE_PCT,
    f'Gini > {TARGET_GINI:.2f}':                         metrics['gini_coefficient'] > TARGET_GINI,
    f'Top decile lift > {TARGET_LIFT:.1f}x':             metrics['top_decile_lift'] > TARGET_LIFT,
    f'Calibration error < {TARGET_CAL_ERROR:.2f}':       metrics['calibration_error'] < TARGET_CAL_ERROR,
}

rprint('\n[bold]Metric Targets (holdout-window aligned):[/bold]')
for target, passed in targets_met.items():
    status = '[bold green]PASS[/bold green]' if passed else '[bold red]FAIL[/bold red]'
    rprint(f'  {status}  {target}')

all_passed = all(targets_met.values())
rprint(f'\nAll targets met: {"[bold green]YES[/bold green]" if all_passed else "[bold yellow]PARTIAL[/bold yellow]"}')


Validating on holdout (window-aligned metrics)...

2026-04-24 15:41:24.830 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:41:24.840 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 3429 customers (freq>0: 1860)
2026-04-24 15:41:24.857 | INFO     | backend.ml.bgnbd_model:validate:535 - Validating on 1875 customers in holdout
2026-04-24 15:41:24.878 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:41:24.880 | INFO     | backend.ml.bgnbd_model:predict:396 - Predicting LTV for 2708 customers…
2026-04-24 15:41:24.914 | DEBUG    | backend.ml.bgnbd_model:_compute_confidence_intervals:461 - Computing 100 bootstrap CI samples…
2026-04-24 15:41:26.716 | INFO     | backend.ml.bgnbd_model:predict:444 - Predictions complete — mean LTV_36m: £9208.84, median: £5861.84
2026-04-24 15:41:26.728 | WARNING  | backend.ml.bgnbd_model:validate:587 - ac

Metric Targets (holdout-window aligned):

PASS  R^2 frequency > 0.65

PASS  MAE holdout < 100% of mean

PASS  Gini > 0.65

PASS  Top decile lift > 3.0x

PASS  Calibration error < 1.00

All targets met: YES

## 6. Calibration Plots

In [10]:
# Predicted vs actual purchases (frequency buckets), aligned to holdout window length
bgf = model.bgf
if bgf is None:
    raise RuntimeError('BG/NBD model is not fitted.')
cal_pd = polars_rfm_to_pandas(rfm_df)
hold_pd = polars_rfm_to_pandas(holdout_rfm_df)
common_idx = cal_pd.index.intersection(hold_pd.index)
cal_pd = cal_pd.loc[common_idx].copy()
hold_pd = hold_pd.loc[common_idx].copy()
cal_pd['predicted_holdout'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    metrics['holdout_days'], cal_pd['frequency'], cal_pd['recency'], cal_pd['T']
)
cal_pd['actual_holdout'] = hold_pd['frequency'].values
n_buckets = 7
cal_pd['frequency_bucket'] = pd.qcut(
    cal_pd['frequency'].rank(method='first'), q=n_buckets, duplicates='drop'
).astype(str)
cal_data_pd = (
    cal_pd.groupby('frequency_bucket', as_index=False)
    .agg(
        predicted_purchases_avg=('predicted_holdout', 'mean'),
        actual_purchases_avg=('actual_holdout', 'mean'),
        customer_count=('actual_holdout', 'size'),
    )
)
cal_data = pl.from_pandas(cal_data_pd)
display(cal_data)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cal_data['frequency_bucket'].to_list(),
    y=cal_data['actual_purchases_avg'].to_list(),
    name='Actual', marker_color='steelblue', opacity=0.7
))
fig.add_trace(go.Scatter(
    x=cal_data['frequency_bucket'].to_list(),
    y=cal_data['predicted_purchases_avg'].to_list(),
    mode='lines+markers', name='Predicted (BG/NBD)',
    line=dict(color='crimson', width=2)
))
fig.update_layout(
    title='BG/NBD Calibration: Predicted vs Actual Purchases by Frequency Bucket',
    xaxis_title='Calibration Frequency',
    yaxis_title='Avg Holdout Purchases',
    height=400,
)
fig.show()


2026-04-24 15:41:32.280 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:41:32.295 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 3429 customers (freq>0: 1860)


frequency_bucket,predicted_purchases_avg,actual_purchases_avg,customer_count
str,f64,f64,i64
"""(0.999, 268.714]""",0.641135,0.925373,268
"""(1071.857, 1339.571]""",2.40557,1.761194,268
"""(1339.571, 1607.286]""",3.340561,2.865672,268
"""(1607.286, 1875.0]""",7.527449,7.567164,268
"""(268.714, 536.429]""",0.645567,1.011194,268
"""(536.429, 804.143]""",1.062548,1.175373,268
"""(804.143, 1071.857]""",1.952218,1.520599,267


## 7. Probability-Alive Matrix

In [11]:
# P(alive) heatmap - frequency vs recency
p_alive_matrix = model.get_probability_alive_matrix(
    max_frequency=50, max_recency_days=365, t_days=365, step=5
)

# Polars pivot API differs by version ('on' vs 'columns'); pick at runtime.
import inspect
from typing import Any, cast

pivot_fn = cast(Any, p_alive_matrix.pivot)
pivot_params = set(inspect.signature(pl.DataFrame.pivot).parameters)
if 'on' in pivot_params:
    pivot = pivot_fn(
        on='recency_days',
        values='p_alive',
        index='frequency',
        aggregate_function='first',
    ).sort('frequency')
else:
    pivot = pivot_fn(
        columns='recency_days',
        values='p_alive',
        index='frequency',
        aggregate_function='first',
    ).sort('frequency')

freq_vals = pivot['frequency'].to_list()
rec_cols  = [c for c in pivot.columns if c != 'frequency']
z = pivot.select(rec_cols).to_numpy()

fig = go.Figure(go.Heatmap(
    z=z,
    x=rec_cols,
    y=[str(f) for f in freq_vals],
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    colorbar=dict(title='P(Alive)'),
))
fig.update_layout(
    title='Probability Alive Matrix: Frequency x Recency',
    xaxis_title='Recency (days since last purchase)',
    yaxis_title='Calibration Frequency',
    height=500,
)
fig.show()

print('\nKey insights:')
print('  High frequency + low recency = high P(alive) -> ideal for retention')
print('  High frequency + high recency = low P(alive) -> likely churned')
print('  Low frequency + low recency = uncertain -> early-stage customer')



Key insights:
  High frequency + low recency = high P(alive) -> ideal for retention
  High frequency + high recency = low P(alive) -> likely churned
  Low frequency + low recency = uncertain -> early-stage customer


## 8. LTV Predictions per Customer

In [12]:
rprint('[bold blue]Generating LTV predictions for all customers...[/bold blue]')
predictions = model.predict(rfm_df, n_bootstrap=200)

print(f'\nPredictions shape: {predictions.shape}')
print(f'Columns: {predictions.columns}')

# Summary stats
print('\nLTV Summary Statistics:')
display(predictions.select(['ltv_12m', 'ltv_24m', 'ltv_36m', 'probability_alive']).describe())


Generating LTV predictions for all customers...

2026-04-24 15:41:35.645 | DEBUG    | backend.ml.bgnbd_model:polars_rfm_to_pandas:103 - Polars→Pandas conversion: 2708 customers (freq>0: 1327)
2026-04-24 15:41:35.647 | INFO     | backend.ml.bgnbd_model:predict:396 - Predicting LTV for 2708 customers…
2026-04-24 15:41:35.692 | DEBUG    | backend.ml.bgnbd_model:_compute_confidence_intervals:461 - Computing 200 bootstrap CI samples…
2026-04-24 15:41:38.509 | INFO     | backend.ml.bgnbd_model:predict:444 - Predictions complete — mean LTV_36m: £9208.84, median: £5861.84



Predictions shape: (2708, 17)
Columns: ['customer_id', 'frequency', 'recency', 'T', 'monetary_value', 'expected_purchases_365d', 'expected_purchases_730d', 'expected_purchases_1095d', 'probability_alive', 'expected_avg_profit', 'ltv_12m', 'ltv_24m', 'ltv_36m', 'ltv_12m_lower', 'ltv_12m_upper', 'ltv_36m_lower', 'ltv_36m_upper']

LTV Summary Statistics:


statistic,ltv_12m,ltv_24m,ltv_36m,probability_alive
str,f64,f64,f64,f64
"""count""",1603.0,1327.0,1327.0,2708.0
"""null_count""",1105.0,1381.0,1381.0,0.0
"""mean""",936.648098,4284.477329,9208.84413,1.0
"""std""",1698.352751,6919.676018,14872.810142,2.2949e-9
"""min""",2.240139,83.819196,180.156844,1.0
"""25%""",279.686824,1662.822299,3573.988187,1.0
"""50%""",577.810582,2727.259413,5861.836789,1.0
"""75%""",1072.281978,4641.11422,9975.381861,1.0
"""max""",38336.484144,146375.666731,314612.634337,1.0


In [13]:
# Preview top customers
top_customers = predictions.sort('ltv_36m', descending=True).head(10)
display(top_customers.select([
    'customer_id', 'frequency', 'probability_alive',
    'expected_avg_profit', 'ltv_12m', 'ltv_24m', 'ltv_36m',
    'ltv_36m_lower', 'ltv_36m_upper'
]))

ltv36_max = _as_float(predictions["ltv_36m"].max())
ltv36_mean = _as_float(predictions["ltv_36m"].mean())
ltv36_median = _as_float(predictions["ltv_36m"].median())
print(f'\nTop customer LTV 36m: £{ltv36_max:.2f}')
print(f'Mean LTV 36m:         £{ltv36_mean:.2f}')
print(f'Median LTV 36m:       £{ltv36_median:.2f}')


customer_id,frequency,probability_alive,expected_avg_profit,ltv_12m,ltv_24m,ltv_36m,ltv_36m_lower,ltv_36m_upper
str,f64,f64,f64,f64,f64,f64,f64,f64
"""16738""",0.0,1.0,3.75,null,null,null,null,null
"""17932""",0.0,1.0,589.01,null,null,null,null,null
"""14813""",0.0,1.0,77.715,11.766907,null,null,null,null
"""17102""",0.0,1.0,25.5,null,null,null,null,null
"""14998""",0.0,1.0,183.64,null,null,null,null,null
"""17631""",0.0,1.0,302.55,null,null,null,null,null
"""13934""",0.0,1.0,922.1,null,null,null,null,null
"""15907""",0.0,1.0,251.18,null,null,null,null,null
"""16270""",0.0,1.0,1141.24,186.578105,null,null,null,null



Top customer LTV 36m: £314612.63
Mean LTV 36m:         £9208.84
Median LTV 36m:       £5861.84


## 9. LTV Distribution + Segmentation Preview

In [14]:
# LTV distribution
p99_q = predictions['ltv_36m'].quantile(0.99)
p99 = _as_float(p99_q, _as_float(predictions['ltv_36m'].max()))
pred_trimmed = predictions.filter(pl.col('ltv_36m') <= p99)

fig = make_subplots(rows=1, cols=3, subplot_titles=['LTV 12m', 'LTV 24m', 'LTV 36m'])
for i, col in enumerate(['ltv_12m', 'ltv_24m', 'ltv_36m'], start=1):
    vals = pred_trimmed[col].to_list()
    fig.add_trace(go.Histogram(x=vals, nbinsx=60, showlegend=False), row=1, col=i)
fig.update_layout(height=350, title='BG/NBD Predicted LTV Distributions')
fig.show()


In [15]:
# Apply LTV segmentation (from Phase 5 plan — preview)
SEGMENT_THRESHOLDS = {
    'champions':   10_000,
    'high_value':  5_000,
    'medium_value':1_000,
}
CAC_PCTS = {
    'champions':    0.50,
    'high_value':   0.40,
    'medium_value': 0.30,
    'low_value':    0.20,
}

predictions_segmented = predictions.with_columns(
    pl.when(pl.col('ltv_36m') > SEGMENT_THRESHOLDS['champions']).then(pl.lit('champions'))
    .when(pl.col('ltv_36m')   > SEGMENT_THRESHOLDS['high_value']).then(pl.lit('high_value'))
    .when(pl.col('ltv_36m')   > SEGMENT_THRESHOLDS['medium_value']).then(pl.lit('medium_value'))
    .otherwise(pl.lit('low_value'))
    .alias('segment')
)

segment_dist = (
    predictions_segmented
    .group_by('segment')
    .agg(
        pl.len().alias('count'),
        pl.col('ltv_36m').mean().alias('avg_ltv_36m'),
        pl.col('ltv_36m').sum().alias('total_ltv_36m'),
        pl.col('probability_alive').mean().alias('avg_p_alive'),
    )
    .sort('avg_ltv_36m', descending=True)
)
display(segment_dist)


segment,count,avg_ltv_36m,total_ltv_36m,avg_p_alive
str,u32,f64,f64,f64
"""champions""",329,22124.128294,7.2788e6,1.0
"""high_value""",444,7133.040628,3.1671e6,1.0
"""medium_value""",547,3235.551431,1.7698e6,1.0
"""low_value""",1388,625.897152,4381.280065,1.0


In [16]:
# Segment pie chart
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Customer Count by Segment', 'Revenue Share by Segment'],
)
fig.add_trace(go.Pie(
    labels=segment_dist['segment'].to_list(),
    values=segment_dist['count'].to_list(),
    hole=0.4,
), row=1, col=1)
fig.add_trace(go.Pie(
    labels=segment_dist['segment'].to_list(),
    values=segment_dist['total_ltv_36m'].to_list(),
    hole=0.4,
), row=1, col=2)
fig.update_layout(height=400, title='LTV Segment Distribution')
fig.show()

# Revenue concentration check
total_ltv = _as_float(segment_dist['total_ltv_36m'].sum())
champ_ltv = _as_float(segment_dist.filter(pl.col('segment') == 'champions')['total_ltv_36m'].sum())
share = (100 * champ_ltv / total_ltv) if total_ltv > 0 else 0.0
print(f'Champions share of total 36m LTV: {share:.1f}%')


Champions share of total 36m LTV: 59.6%


In [17]:
# P(alive) vs LTV scatter
sample = predictions_segmented.sample(min(2000, len(predictions_segmented)))
fig = px.scatter(
    sample.to_pandas(),
    x='probability_alive',
    y='ltv_36m',
    color='segment',
    opacity=0.5,
    title='Probability Alive vs Predicted 36m LTV',
    labels={'probability_alive': 'P(Alive)', 'ltv_36m': 'LTV 36m (£)'},
    category_orders={'segment': ['champions', 'high_value', 'medium_value', 'low_value']},
)
fig.show()


In [18]:
# Confidence interval width analysis
predictions_with_ci = predictions.with_columns(
    (pl.col("ltv_36m_upper") - pl.col("ltv_36m_lower")).alias("ci_width_36m")
)

plot_df = predictions_with_ci.filter(pl.col("ltv_36m") <= p99)

# sample safely from filtered rows
n = min(2000, len(plot_df))
plot_df = plot_df if n == len(plot_df) else plot_df.sample(n=n, seed=42)

fig = px.scatter(
    plot_df.to_pandas(),
    x="ltv_36m",
    y="ci_width_36m",
    opacity=0.4,
    title="LTV 36m Prediction vs Confidence Interval Width",
    labels={"ltv_36m": "LTV 36m (£)", "ci_width_36m": "CI Width (90%)"},
)
fig.show()


## 10. W&B Logging

In [21]:
wandb_run_id = None

if USE_WANDB:
    tracker = WandbTracker(
        project=settings.WANDB_PROJECT,
        name='week2_bgnbd_uci',
        tags=['week2', 'bgnbd', 'gamma_gamma', 'uci'],
        config={
            'penalizer_coef':     best_penalizer,
            'observation_months': OBSERVATION_MONTHS,
            'holdout_months':     HOLDOUT_MONTHS,
            'observation_end':    str(obs_end),
            'n_customers':        len(rfm_df),
            'margin':             0.20,
            'discount_rate':      0.10,
        },
        enabled=USE_WANDB,
    )
    
    with tracker:
        tracker.log_params(model)
        tracker.log_metrics(metrics)
        tracker.log_calibration_plot(cal_data)
        tracker.log_ltv_distribution(predictions, 'ltv_36m')
        tracker.log_ltv_distribution(predictions, 'ltv_12m')
        tracker.log_probability_alive_matrix(p_alive_matrix)
        tracker.log_predictions_table(predictions_segmented, max_rows=500)
        if RUN_HYPEROPT:
            tracker.log_grid_search_results(grid_results)
        tracker.alert_metric_target(metrics)
        
        if tracker._run:
            wandb_run_id = tracker._run.id
    
    print(f'✓ W&B run logged (run_id={wandb_run_id})')
else:
    print('W&B logging skipped')


W&B logging skipped


## 11. Save to Supabase

In [22]:
if SAVE_TO_DB:
    import uuid
    from datetime import datetime, timezone
    
    db = SupabaseClient(use_service_role=True)
    assert db.health_check(), 'DB health check failed — check .env'
    
    run_id = str(uuid.uuid4())[:8]
    
    # Save model parameters
    print('Saving model parameters...')
    model.save_params(db, pipeline_run_id=run_id, wandb_run_id=wandb_run_id)
    
    # Save per-customer predictions
    print('Saving predictions...')
    n_saved = model.save_predictions(predictions_segmented, db)
    print(f'✓ Saved {n_saved:,} predictions')
    
    # Save probability-alive matrix
    print('Saving probability-alive matrix...')
    model.save_probability_alive_matrix(db)
    
    # Save model to disk
    print('Saving model to disk...')
    model.save_to_disk(MODELS_DIR)
    print(f'✓ Model saved to {MODELS_DIR}')
    
    # Log pipeline run
    db.bulk_upsert('pipeline_runs', [{
        'run_id':            run_id,
        'pipeline_name':     'bgnbd_fitting',
        'status':            'success',
        'started_at':        datetime.now(timezone.utc).isoformat(),
        'records_processed': len(predictions),
        'metadata': {
            'model_version':   model.model_version,
            'penalizer':       best_penalizer,
            'metrics':         {k: round(float(v), 4) for k, v in metrics.items() 
                                 if isinstance(v, (int, float))},
            'wandb_run_id':    wandb_run_id,
        },
        'wandb_run_id':      wandb_run_id,
    }], conflict_columns=['run_id'])
    
    print(f'\n✓ All data saved to Supabase (run_id={run_id})')
else:
    print('Skipping DB save (SAVE_TO_DB=False)')
    # Still save model to disk locally
    model.save_to_disk(MODELS_DIR)


2026-04-24 15:41:58.170 | INFO     | backend.ml.bgnbd_model:save_to_disk:854 - Model saved to E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models


Skipping DB save (SAVE_TO_DB=False)


In [23]:
# Final summary
print('\n=== Week 2 Complete - BG/NBD + Gamma-Gamma Summary ===')
print(f'  Model version:          {model.model_version}')
print(f'  Customers scored:       {len(predictions):,}')
print(f'  Penalizer coef:         {best_penalizer}')
bgnbd_ll = model._fit_metrics.get('bgnbd_log_likelihood')
gg_ll = model._fit_metrics.get('gg_log_likelihood')
bgnbd_ll_txt = f'{bgnbd_ll:.4f}' if isinstance(bgnbd_ll, (int, float)) and np.isfinite(bgnbd_ll) else 'N/A (not exposed by lifetimes)'
gg_ll_txt = f'{gg_ll:.4f}' if isinstance(gg_ll, (int, float)) and np.isfinite(gg_ll) else 'N/A (not exposed by lifetimes)'
print(f'  BG/NBD log-likelihood:  {bgnbd_ll_txt}')
print(f'  GG log-likelihood:      {gg_ll_txt}')
print(f'  Holdout window:         {metrics["holdout_days"]} days (~{metrics["holdout_months"]}m)')
print(f'  R^2 (frequency):        {metrics["r2_frequency"]:.4f}  (target > {TARGET_R2_FREQUENCY:.2f})')
print(f'  MAE LTV holdout:        GBP {metrics["mae_ltv_12m"]:.2f}  ({100*metrics["mae_pct_12m"]:.1f}% of mean, target < {100*TARGET_MAE_PCT:.0f}%)')
print(f'  Gini:                   {metrics["gini_coefficient"]:.4f}  (target > {TARGET_GINI:.2f})')
print(f'  Top decile lift:        {metrics["top_decile_lift"]:.2f}x  (target > {TARGET_LIFT:.1f}x)')
print(f'  Calibration error:      {metrics["calibration_error"]:.4f}  (target < {TARGET_CAL_ERROR:.2f})')
mean_ltv_36m = _as_float(predictions["ltv_36m"].mean())
print(f'  Mean LTV 36m:           GBP {mean_ltv_36m:.2f}')
print(f'\nTargets met: {sum(targets_met.values())}/{len(targets_met)}')



=== Week 2 Complete - BG/NBD + Gamma-Gamma Summary ===
  Model version:          bgnbd_uci_v1
  Customers scored:       2,708
  Penalizer coef:         0.1
  BG/NBD log-likelihood:  N/A (not exposed by lifetimes)
  GG log-likelihood:      N/A (not exposed by lifetimes)
  Holdout window:         180 days (~6m)
  R^2 (frequency):        0.6768  (target > 0.65)
  MAE LTV holdout:        GBP 1456.39  (93.2% of mean, target < 100%)
  Gini:                   0.7697  (target > 0.65)
  Top decile lift:        4.96x  (target > 3.0x)
  Calibration error:      0.9135  (target < 1.00)
  Mean LTV 36m:           GBP 9208.84

Targets met: 5/5
